# Exploration: the three ways an experiment goes wrong

None of them are statistical errors. All three are decision errors, and each has a number
that would have caught it.

Everything here is computed by the toolkit; the validation data is simulated on purpose,
because checking whether an implementation does the right arithmetic requires knowing the
answer in advance.

In [1]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd

from abtest.analyse import bayesian, frequentist
from abtest.cuped import adjust
from abtest.design import detectable_effect, sample_size_proportions
from abtest.report import decide, render

## Failure 1: the test never had the power to answer

A team with 20,000 users a day plans "a week or two" and assumes that is enough.

In [2]:
BASELINE = 0.05
rows = []
for days in (3, 7, 14, 30):
    per_variant = 20_000 * days // 2
    rows.append({
        "days": days,
        "users_per_variant": f"{per_variant:,}",
        "smallest_detectable_lift": f"{detectable_effect(BASELINE, per_variant):.1%}",
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
plan = sample_size_proportions(BASELINE, 0.05)
print(f"To detect a 5% relative lift: {plan.per_variant:,} per variant "
      f"= {plan.days_required(20_000):.1f} days")

 days users_per_variant smallest_detectable_lift
    3            30,000                    10.2%
    7            70,000                     6.6%
   14           140,000                     4.7%
   30           300,000                     3.2%

To detect a 5% relative lift: 122,124 per variant = 12.2 days


A one-week test cannot see a 5% improvement. If the feature delivers exactly that, the
readout says "no effect" — and the team ships nothing and learns the wrong lesson.

Sample size scales with `1 / mde²`, so halving the effect you want to catch quadruples the
traffic you need.

In [3]:
for mde in (0.20, 0.10, 0.05, 0.025):
    print(f"  MDE {mde:>5.1%} relative -> {sample_size_proportions(BASELINE, mde).per_variant:>9,} per variant")

  MDE 20.0% relative ->     8,158 per variant
  MDE 10.0% relative ->    31,234 per variant
  MDE  5.0% relative ->   122,124 per variant
  MDE  2.5% relative ->   482,858 per variant


## Failure 2: the p-value gets reported in the Bayesian sentence

They answer different questions, and the difference shows up most when the result is
marginal.

In [4]:
freq = frequentist(1_000, 20_000, 1_090, 20_000)
bayes = bayesian(1_000, 20_000, 1_090, 20_000)
print(f"p-value          {freq.p_value:.4f}   = P(data this extreme | NO effect)")
print(f"P(B beats A)     {bayes.probability_treatment_better:.3f}   = P(effect | THIS data)")
print()
print("'We are 95% confident the variant wins' is the second sentence")
print("attached to the first number.")

p-value          0.0432   = P(data this extreme | NO effect)
P(B beats A)     0.978   = P(effect | THIS data)

'We are 95% confident the variant wins' is the second sentence
attached to the first number.


## Failure 3: shipping something real but too small to matter

Statistical significance is not a decision rule. With enough traffic anything is
significant, including a lift that does not pay for the engineering.

In [5]:
print(render(decide(12_200, 244_000, 12_566, 244_000, minimum_worthwhile_lift=0.05)))

VERDICT: HOLD
  The effect looks real (P = 99.2%) but the lift of 3.0% is below the 5.0% worth shipping for.

Frequentist
  control            5.0000%
  treatment          5.1500%
  relative lift      +3.00%
  p-value            0.0170
  95% CI (absolute)  [+0.0268%, +0.2732%]

Bayesian
  P(treatment better)   99.2%
  expected lift      +3.01%
  95% credible       [+0.53%, +5.54%]
  expected loss      0.000002 conversion rate

Note: the p-value is P(data | no effect). P(treatment better) is
P(effect | data). They are not the same statement and should not be
swapped when the result is written up.


## And the lever that buys traffic you do not have

CUPED subtracts the part of a user's behaviour that was predictable from before the
experiment started. The expected effect is untouched; the noise around it shrinks.

In [6]:
rng = np.random.default_rng(0)
rows = []
for strength in (0.0, 0.3, 0.5, 0.7, 0.9):
    pre = rng.gamma(2, 50, 40_000)
    post = strength * pre + rng.gamma(2, 20, 40_000)
    _, result = adjust(post, pre)
    rows.append({
        "correlation": round(result.correlation, 3),
        "variance_reduction": f"{result.variance_reduction:.1%}",
        "equivalent_traffic": f"{result.effective_sample_multiplier:.2f}x",
    })
print(pd.DataFrame(rows).to_string(index=False))

 correlation variance_reduction equivalent_traffic
       0.007               0.0%              1.00x
       0.591              35.0%              1.54x
       0.779              60.8%              2.55x
       0.867              75.1%              4.02x
       0.913              83.3%              5.99x


The covariate must be measured **before** randomisation. That is what keeps the estimate
unbiased, and using a covariate measured during the experiment is the one way to get CUPED
badly wrong.

At a correlation of 0.87, variance falls about 75% — worth 4x the users. The 12-day test
above becomes a 3-day test, with no change to the estimate itself.